# Part 5 · Agent Substrate: your agents in a gVisor sandbox

Parts 1-4 secured *service* and *agent* traffic. This part is about **where an agent runs**. Solo's kagent **Agent Substrate** runs each agent as an **actor inside a gVisor sandbox** on a pool of pre-warmed workers, with memory snapshots for fast resume. Same `mesh1` cluster; a `SandboxAgent` instead of an ordinary pod.

> **Beta / off by default.** Substrate ships in kagent-enterprise ≥ v0.5.2, disabled unless you turn it on. It runs on kind with **gVisor (runsc)** — no `/dev/kvm` needed (gVisor is a userspace kernel). This is the newest, least-hardened path in the suite; treat it as a preview and rehearse before presenting.

### The substrate components

`substrate-up.sh` turns on kagent's **Agent Substrate** engine (API group `ate.dev`; every component is prefixed `ate*`). What it adds to the `kagent` namespace:

| Component | Role |
|---|---|
| **SandboxAgent** (CRD) | the agent you deploy — runs as a gVisor actor, not an ordinary pod |
| **WorkerPool** (`ate.dev`) | pool of pre-warmed gVisor workers (`ateom-gvisor` image) |
| **ate-api-server** | control-plane gRPC API (`kagent-api.kagent.svc:443`) — manages actors, resolves their env |
| **ate-controller** | reconciles `ActorTemplate`s / golden actors |
| **atelet** (DaemonSet) | downloads `runsc` and runs actors under gVisor on each node |
| **atenet-router** | actor networking |
| **valkey** | actor / worker state store |


## Connect · run this first

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
export CTX KAGENT_NS
export KENT_CRDS_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise-crds"
export KENT_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise"
export KAGENT_ENT_VERSION="${KAGENT_ENT_VERSION:-0.5.2}"
echo "context: $CTX ; kagent-enterprise target: $KAGENT_ENT_VERSION"
kubectl --context $CTX version -o json 2>/dev/null | python3 -c "import sys,json;print('k8s:',json.load(sys.stdin)['serverVersion']['gitVersion'],'(substrate needs >=1.33)')" 2>/dev/null || true

## How substrate gets enabled (setup, not this notebook)

Substrate is turned on at **cluster setup**, not from here: run setup with `ENABLE_SUBSTRATE=true`, or add it to a running cluster with `./demo-scripts/substrate-up.sh`. That script **bumps the shared kagent to v0.5.2** (Part 4's AgentRegistry runs v0.4.3), so run it when you are not mid-Part-4. Needs Kubernetes ≥ 1.33 (mesh1 is v1.35 ✓) and, on Apple Silicon, the arm64 `ateom-gvisor` image. **This notebook only demos the capability.**


In [ ]:
: "${CTX:=kind-mesh1}"
# ⚠️ ONE-TIME: enable Agent Substrate. This bumps the SHARED kagent release to v0.5.2
#    (Part 4's AgentRegistry runs v0.4.3), so it changes Part 4's kagent too. It is
#    idempotent and already done if you ran setup with ENABLE_SUBSTRATE=true. ~several min.
bash "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/substrate-up.sh"

## 5.1 · Beat 1 — prove the agent runs in a gVisor sandbox

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="260" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Beat 1 · Where your agent actually runs: a gVisor-sandboxed actor</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="14" y="66" width="116" height="46" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.5"/><text x="72" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">SandboxAgent</text><text x="72" y="101" text-anchor="middle" font-size="8" fill="#4338ca">(kagent CRD)</text><line x1="130" y1="89" x2="168" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><rect x="170" y="66" width="120" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="230" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">kagent-</text><text x="230" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">controller</text><line x1="290" y1="89" x2="330" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="310" y="80" text-anchor="middle" font-size="7.5" fill="#475569">ActorTemplate</text><rect x="332" y="66" width="110" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="387" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">ate-api-</text><text x="387" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">server</text><line x1="442" y1="89" x2="468" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="455" y="80" text-anchor="middle" font-size="7.5" fill="#475569">bind</text><rect x="470" y="52" width="236" height="120" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="588" y="70" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool worker · ateom-gvisor</text><rect x="486" y="84" width="204" height="74" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="588" y="102" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">gVisor sandbox (runsc)</text><text x="588" y="120" text-anchor="middle" font-size="10" font-weight="700" fill="#92400e">ADK actor</text><text x="588" y="138" text-anchor="middle" font-size="8" fill="#b45309">guest kernel ≠ host kernel</text><rect x="300" y="196" width="220" height="40" rx="8" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.4"/><text x="410" y="214" text-anchor="middle" font-size="9" font-weight="700" fill="#5b21b6">kagent-atelet (DaemonSet)</text><text x="410" y="228" text-anchor="middle" font-size="8" fill="#6d28d9">installs runsc on the node</text><line x1="520" y1="206" x2="588" y2="172" stroke="#8b5cf6" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#v)"/><text x="566" y="192" text-anchor="middle" font-size="7.5" fill="#6d28d9">runsc</text><text x="360" y="252" text-anchor="middle" font-size="10" fill="#64748b">A SandboxAgent → ActorTemplate → bound onto a pooled gVisor worker; the agent runs in a runsc sandbox with its own guest kernel.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
# substrate must be enabled first (the ‘Enable substrate’ cell above, or setup ENABLE_SUBSTRATE=true).
# deploy a SandboxAgent — it runs as a gVisor actor on the WorkerPool, not an ordinary pod.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo, namespace: kagent }
spec:
  type: Declarative
  description: Minimal Go ADK SandboxAgent running on Agent Substrate (gVisor).
  declarative:
    runtime: go                       # go = faster startup; python = full feature set
    modelConfig: default-model-config # auto-created from the anthropic provider
    systemMessage: "You are a helpful assistant running inside a gVisor-sandboxed actor."
  substrate:
    workerPoolRef: { name: kagent-default }
EOF
kubectl --context $CTX -n $KAGENT_NS get sandboxagent substrate-demo

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
printf '%s== the pool worker runs the gVisor ateom image ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get pods -l kagent.dev/worker-pool=kagent-default -o jsonpath='{.items[*].spec.containers[*].image}'; echo
printf '%s== the WorkerPool declares sandboxClass gvisor ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get workerpool kagent-default -o jsonpath='{.spec.sandboxClass}'; echo
printf '%s== the atelet DaemonSet installed runsc on the node ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get ds kagent-atelet
kubectl --context $CTX -n $KAGENT_NS logs ds/kagent-atelet --tail=20 | grep -i runsc || true
echo
echo "Money shot (if the runtime exposes a shell tool): chat the agent to run  dmesg | head  and  cat /proc/version"
echo "— a gVisor guest reports a synthetic kernel and a whimsical gVisor boot log, not the host 6.x kernel."

## 5.2 · Beat 2 — warm pool vs cold bind

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 238" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="238" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Beat 2 · Warm pool vs cold bind</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="20" y="58" width="190" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="115" y="78" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool: 2 ready workers</text><text x="115" y="94" text-anchor="middle" font-size="8" fill="#166534">warm, golden resident</text><line x1="210" y1="84" x2="300" y2="84" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="255" y="76" text-anchor="middle" font-size="7.5" fill="#166534">chat</text><rect x="302" y="64" width="150" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.4"/><text x="377" y="80" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">bind to warm worker</text><text x="377" y="94" text-anchor="middle" font-size="7.5" fill="#166534">resume from golden</text><rect x="470" y="64" width="90" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="515" y="88" text-anchor="middle" font-size="13" font-weight="700" fill="#14532d">~1s</text><rect x="20" y="150" width="190" height="52" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/><text x="115" y="170" text-anchor="middle" font-size="9" font-weight="700" fill="#334155">WorkerPool: 0 (scaled down)</text><text x="115" y="186" text-anchor="middle" font-size="8" fill="#475569">cold</text><line x1="210" y1="176" x2="300" y2="176" stroke="#d97706" stroke-width="1.7" marker-end="url(#d)"/><text x="255" y="168" text-anchor="middle" font-size="7.5" fill="#92400e">chat</text><rect x="302" y="156" width="150" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="377" y="171" text-anchor="middle" font-size="7.5" font-weight="700" fill="#7c2d12">schedule pod → ateom</text><text x="377" y="184" text-anchor="middle" font-size="7.5" fill="#92400e">start → runsc spawn → bind</text><rect x="470" y="156" width="140" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="540" y="180" text-anchor="middle" font-size="13" font-weight="700" fill="#7c2d12">~10–40s</text><text x="360" y="224" text-anchor="middle" font-size="10" fill="#64748b">Two live timings: a warm worker resumes fast; a cold pool must schedule a pod, start ateom and spawn the runsc actor. Numbers vary on a laptop.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
# WARM: pool already at 2 ready workers — the actor binds to a pre-warmed worker.
kubectl --context $CTX -n $KAGENT_NS wait workerpool/kagent-default --for=jsonpath='{.status.replicas}'=2 --timeout=120s
echo "warm pool ready — time a first chat to substrate-demo (via kagent A2A / UI) and note the wall clock."
echo "(reuse whatever you use elsewhere to talk to a kagent agent; the bind resumes from the warm golden)"

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
# COLD: scale the pool to 0, then a request forces a fresh worker + runsc actor spawn.
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":0}}'
kubectl --context $CTX -n $KAGENT_NS wait workerpool/kagent-default --for=jsonpath='{.status.replicas}'=0 --timeout=120s
echo "pool cold. Now bring one worker back and time the same chat:"
time ( kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":1}}'; \
       kubectl --context $CTX -n $KAGENT_NS wait workerpool/kagent-default --for=jsonpath='{.status.replicas}'=1 --timeout=300s )
echo "cold path = pod schedule + ateom start + runsc actor spawn + bind. Restore 2 replicas afterwards:"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}'

## 5.3 · Beat 3 — golden actor + snapshot resume (narration)

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Beat 3 · Golden actor + snapshot resume (stretch, narrate-only)</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="280" y="64" width="160" height="54" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="360" y="86" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">golden actor</text><text x="360" y="102" text-anchor="middle" font-size="8" fill="#166534">memory snapshot on pause</text><rect x="500" y="66" width="200" height="50" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="600" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#334155">gs:// snapshot bucket</text><text x="600" y="101" text-anchor="middle" font-size="8" fill="#b91c1c">requires GCS · no local option</text><line x1="440" y1="86" x2="498" y2="86" stroke="#334155" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#n)"/><text x="469" y="78" text-anchor="middle" font-size="7.5" fill="#475569">snapshot</text><rect x="160" y="150" width="240" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="280" y="170" text-anchor="middle" font-size="10" font-weight="700" fill="#1e293b">new actor bind</text><text x="280" y="186" text-anchor="middle" font-size="8.5" fill="#475569">ResumeGoldenActor</text><line x1="360" y1="118" x2="300" y2="148" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="345" y="138" text-anchor="middle" font-size="7.5" fill="#166534">resume</text><rect x="430" y="150" width="270" height="50" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="565" y="169" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">internal engine reconcile</text><text x="565" y="185" text-anchor="middle" font-size="8" fill="#92400e">no operator verb · rehearsal-required</text><text x="360" y="222" text-anchor="middle" font-size="10" fill="#64748b">An idle actor snapshots to storage and resumes from a golden, not a cold boot. No kubectl verb and snapshots need GCS, so narrate this one.</text></svg></div>

There is **no operator verb** for pause/snapshot/resume — `ResumeGoldenActor` is an internal reconcile phase of the substrate engine. The only user-facing knob is where snapshots persist, and it is **GCS-only**:

```yaml
spec:
  substrate:
    workerPoolRef: { name: kagent-default }
    snapshotsConfig:
      location: gs://<your-bucket>/kagent/substrate-demo/   # must be gs:// — no local option
```

So demo the **fast warm resume** from Beat 2 as the visible payoff, and narrate the golden-snapshot machinery. Persisting snapshots across workers needs a real GCS bucket + credentials; skip it for an offline laptop run.

## Tear down

In [ ]:
: "${CTX:=kind-mesh1}" "${KAGENT_NS:=kagent}"
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo --ignore-not-found
# to fully remove substrate (and hand kagent back to Part 4):
#   helm --kube-context $CTX upgrade kagent "$KENT_CHART" -n $KAGENT_NS --reuse-values \
#     --set substrate.enabled=false --set substrateWorkerPool.create=false --set controller.substrate.enabled=false
#   kubectl --context $CTX -n $KAGENT_NS delete workerpool kagent-default --ignore-not-found